# Align FOV positions across microscopes

Re-image the **same FOVs** on a second microscope after physically moving the
stage insert. Assumes the X/Y axis directions are preserved (no rotation) and
the sample is rigid, so the two stage coordinate systems differ only by an
**isotropic similarity transform** — one uniform scale plus a translation:

$$q = s\,p + (t_x, t_y)$$

The transform is found by **overlapping the two tissue-boundary polygons** (one
drawn on each microscope): a closed-form centroid/area match initialises the
fit, which is then refined by maximising the polygon intersection-over-union
(IoU). The transform is finally applied to the source FOV positions to produce
the target-microscope positions file.

**Inputs** (in `SAMPLE_DIR/positions/`):
- `boundary_positions_{SOURCE}.txt` — boundary drawn on the source scope
- `boundary_positions_{TARGET}.txt` — boundary drawn on the target scope
- `positions_{SAMPLE_NAME}.txt` — FOV positions used on the source scope

**Output**: `positions_{SAMPLE_NAME}_{TARGET}.txt` — FOV positions for the target scope.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.io            import load_positions, save_positions_array
from MERci.acquisition.alignment import load_boundary_polygon, fit_isotropic_alignment

POSITIONS_DIR = SAMPLE_DIR / "positions"
SAMPLE_NAME   = SAMPLE_DIR.name
print(f"SAMPLE_DIR  : {SAMPLE_DIR}")
print(f"SAMPLE_NAME : {SAMPLE_NAME}")

In [ ]:
# ── Parameters ───────────────────────────────────────────────
# Microscope labels (used only to build the default file names below).
SOURCE_LABEL = "mf4"    # where the run was already imaged (epifluorescence)
TARGET_LABEL = "mf2"    # where you want to re-image the same FOVs (confocal)

# Boundary polygons (comma-separated x,y per line; one vertex per line).
SOURCE_BOUNDARY = POSITIONS_DIR / f"boundary_positions_{SOURCE_LABEL}.txt"
TARGET_BOUNDARY = POSITIONS_DIR / f"boundary_positions_{TARGET_LABEL}.txt"

# Source FOV positions (the file used to acquire on the source scope).
SOURCE_POSITIONS = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt"

# Output FOV positions for the target scope.
TARGET_POSITIONS = POSITIONS_DIR / f"positions_{SAMPLE_NAME}_{TARGET_LABEL}.txt"

REFINE = True   # maximise polygon IoU after the closed-form centroid/area init

for p in (SOURCE_BOUNDARY, TARGET_BOUNDARY, SOURCE_POSITIONS):
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

In [ ]:
# ── Load boundaries and source FOV positions ──────────────────────────
src_poly = load_boundary_polygon(SOURCE_BOUNDARY)
tgt_poly = load_boundary_polygon(TARGET_BOUNDARY)

pos_dict     = load_positions(SOURCE_POSITIONS)              # {fov_id: (x, y)}
src_fovs     = np.array([pos_dict[i] for i in sorted(pos_dict)], dtype=float)

print(f"Source boundary : {len(src_poly.exterior.coords) - 1:4d} vertices, area={src_poly.area:,.0f}")
print(f"Target boundary : {len(tgt_poly.exterior.coords) - 1:4d} vertices, area={tgt_poly.area:,.0f}")
print(f"Source FOVs     : {len(src_fovs)}")

In [ ]:
# ── Fit the isotropic transform (scale + translation, no rotation) ────────
fit = fit_isotropic_alignment(src_poly, tgt_poly, refine=REFINE)

print(f"scale        : {fit.scale:.5f}")
print(f"translation  : ({fit.tx:,.2f}, {fit.ty:,.2f})")
print(f"IoU (init)   : {fit.iou_init:.4f}")
print(f"IoU (final)  : {fit.iou:.4f}   {'(refined)' if fit.refined else '(closed-form)'}")
if fit.iou < 0.8:
    print("\n⚠  Low overlap — check that the boundaries are from the same sample,")
    print("   that the axes are not flipped/rotated, and that both files are valid.")

In [ ]:
# ── Apply the transform to the FOV positions ─────────────────────────
tgt_fovs = fit.transform_points(src_fovs)

print(f"Mapped {len(tgt_fovs)} FOV positions into the {TARGET_LABEL} coordinate system.")
print(f"  source x ∈ [{src_fovs[:,0].min():.0f}, {src_fovs[:,0].max():.0f}], "
      f"y ∈ [{src_fovs[:,1].min():.0f}, {src_fovs[:,1].max():.0f}]")
print(f"  target x ∈ [{tgt_fovs[:,0].min():.0f}, {tgt_fovs[:,0].max():.0f}], "
      f"y ∈ [{tgt_fovs[:,1].min():.0f}, {tgt_fovs[:,1].max():.0f}]")

In [ ]:
# ── Visualise the alignment ─────────────────────────────────────
def _ring(poly):
    return np.asarray(poly.exterior.coords)

src_ring  = _ring(src_poly)
tgt_ring  = _ring(tgt_poly)
warp_ring = _ring(fit.transform_polygon(src_poly))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 6))

ax0.plot(src_ring[:, 0], src_ring[:, 1], "-", color="tab:blue", label=f"{SOURCE_LABEL} boundary")
ax0.plot(tgt_ring[:, 0], tgt_ring[:, 1], "-", color="tab:orange", label=f"{TARGET_LABEL} boundary")
ax0.scatter(src_fovs[:, 0], src_fovs[:, 1], s=6, color="tab:blue", alpha=0.5, label=f"{SOURCE_LABEL} FOVs")
ax0.set_title("Before: raw coordinates")
ax0.legend(loc="best", fontsize=8)
ax0.set_aspect("equal"); ax0.grid(alpha=0.3)

ax1.plot(tgt_ring[:, 0], tgt_ring[:, 1], "-", color="tab:orange", lw=2, label=f"{TARGET_LABEL} boundary")
ax1.plot(warp_ring[:, 0], warp_ring[:, 1], "--", color="tab:blue", label=f"{SOURCE_LABEL}→{TARGET_LABEL} boundary")
ax1.scatter(tgt_fovs[:, 0], tgt_fovs[:, 1], s=6, color="tab:blue", alpha=0.6, label="mapped FOVs")
ax1.set_title(f"After: IoU = {fit.iou:.3f}")
ax1.legend(loc="best", fontsize=8)
ax1.set_aspect("equal"); ax1.grid(alpha=0.3)

fig.suptitle(f"{SOURCE_LABEL} → {TARGET_LABEL}  (scale={fit.scale:.4f}, t=({fit.tx:.0f}, {fit.ty:.0f}))")
fig.tight_layout()

plot_path = POSITIONS_DIR / f"alignment_{SAMPLE_NAME}_{SOURCE_LABEL}_to_{TARGET_LABEL}.png"
fig.savefig(plot_path, dpi=150)
print(f"Saved diagnostic plot: {plot_path}")
plt.show()

In [ ]:
# ── Save the target-microscope FOV positions ─────────────────────────
save_positions_array(tgt_fovs, TARGET_POSITIONS)
print(f"Saved {len(tgt_fovs)} positions: {TARGET_POSITIONS}")